# Bacteriophage Surface Walk Simulation
This notebook simulates bacteriophage walks a host suface using 3D diffusion and residence time logic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
P = 1e+5
B = 1
nsteps = 1000
trials = 1

In [ ]:
avg_res_times=np.zeros(trials,1);     #store average resident times per trial
avg_kernel=np.zeros(trials,1);          #store average kernel times per trial  

data1 = np.zeros(trials,1)
data2 = np.zeros(trials,1)

# dimensionality parameters (1=on, 0=off). The sets the viewing angle
D_para_y = 1                                                       #y-dir
D_para_z = 1                                                       #z-dir  

#sim parameters
omega = 1                                          # Bacteria Tumble rate
D0 = 2.54                                          # Diffusion Constant
nu = 1 # 0.82;                                       # Diffusion Exponent
tmin = 0.001                                   # power-law gamma parameter
dt=0.01 

# Define parameters
trials = 50
v0 = np.linspace(0.0, 30.0, trials)
dt = 0.01
omega = 1  # Tumble rate

for indx in range(trials):  # Python uses 0-based indexing
    print(indx)  # Print the index if needed for debug

    res_times = []  # storage for distributions of residence times
    dep_vels = []   # storage for distributions of deposition velocities

    # Parameters ----------------------------
    v = v0[indx]
    Rb = 1
    Rp = 0.1
    td = 0 * dt
    Time = 0
    LX = 10
    LY = 10

    # Simulation space
    x_left = -LX / 2
    x_right = LX / 2
    y_left = -LY / 2
    y_right = LY / 2
    z_left = -LX / 2  # Assuming cubic domain
    z_right = LX / 2

    V = LX * LX * LX
    Rcrit = Rb + Rp  # "soft core"
    x_old = 0
    y_old = 0
    z_old = 0

    # Bacteria runtime
    tau = 0  # or: -np.log(np.random.rand()) / omega
    run_time = Time + tau


In [ ]:
# ==========================================================================
#                         System Initialization
# ==========================================================================

# Simulation Parameters
kernel_new = np.zeros((1, nsteps))
kernel_avg = np.zeros((1, nsteps))


# Bacteria Initialization: MatB columns: [x, y, z, vx, vy, vz, transition_timer]
MatB = np.zeros((B, 7))
MatB[:, 3:6] = [-v, 0, 0]  # All bacteria moving in -x direction

# Rcrit-based spatial parameter
RR = x_right - 0 * (Rb + Rp)

# Phage Initialization: MatP columns: [x, y, z, transition_timer, active_state, contact_timer,x_old, y_old, z_old, 
# x_backup, y_backup, z_backup]

MatP = np.zeros((P, 12))

for p in range(P):
    r = 0
    while r < (Rb + Rp):
        x_old, y_old, z_old = [
            ((hi - lo - 2 * Rp) * np.random.rand()) + lo + Rp
            for lo, hi in [(x_left, x_right), (y_left, y_right), (z_left, z_right)]
        ]
        r = np.sqrt(
            (x_old - MatB[0, 0])**2 +
            D_para_y * (y_old - MatB[0, 1])**2 +
            D_para_z * (z_old - MatB[0, 2])**2
        )

    MatP[p, 0:3] = x_old, D_para_y * y_old, D_para_z * z_old  # new position
    MatP[p, 3:6] = 0, 1, 0                                    # timers and state
    MatP[p, 6:9] = x_old, y_old, z_old                        # old positions
    MatP[p, 9:12] = x_old, y_old, z_old                       # backups


In [ ]:

# ==========================================================================
#                         Phage Distribution Histogram
# ==========================================================================

# Radial distance from origin
r = np.linalg.norm(MatP[:, 0:3], axis=1)

# Exclude distant phages
r = r[r <= 2.5]

# Histogram and shell volume normalization
n, edges = np.histogram(r, bins=40)
r_centers = 0.5 * (edges[:-1] + edges[1:])
dr = r_centers[1] - r_centers[0]
volumes = (4 / 3) * np.pi * ((r_centers + dr)**3 - (r_centers - dr)**3)

# Normalized radial density plot
plt.figure(figsize=(6, 6))
plt.plot(r_centers, n / (np.sum(n) * volumes), 'k--', linewidth=2)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlabel('Radius')
plt.ylabel('Normalized Concentration')
plt.title('Phage Concentration Distribution')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================================
#                    Class-Based Simulation Execution
# ==========================================================================

# Initialize simulator object
sim = PhageSimulator(B=B, P=P, params={
    "D0": D0,
    "Rb": Rb,
    "Rp": Rp,
    "dt": dt,
    "tmin": tmin,
    "nu": nu,
    "D_para_y": D_para_y,
    "D_para_z": D_para_z,
    "nsteps": nsteps,
    "omega": omega,
    "LX": LX,
})

# Initialize dynamic variables
n = 1
Time = 0
res_times = []
dep_vels = []
sim.run_time = 0  # Initialize bacteria runtime

while n < sim.params["nsteps"]:
    if n % 100 == 0:
        print(f"Progress: {100 * n / sim.params['nsteps']:.1f}%")

    sim.update_phage_positions(dt=dt)
    sim.resolve_overlap()
    sim.generate_noise(dt=dt, tmin=tmin, nu=nu, time=Time)

    Nenc, Nact, res_times, dep_vels, sim.N_dep = sim.compute_encounters(
        res_times, dep_vels, time=Time
    )

    sim.kernel_new[0, n] = (Nenc / Nact) * (sim.V / dt)
    if n > 500:
        sim.kernel_avg[0, n] = np.mean(sim.kernel_new[0, n - 500:n])

    # Decide when to update bacteria direction
    if 0 * Time >= sim.run_time:
        sim.update_bacteria_direction()

    sim.apply_boundary_conditions()
    sim.resolve_overlap()

    Time += dt
    n += 1

# ==========================================================================
#                        Post-Processing and Save
# ==========================================================================

sim.plot_simulation_state()

# Optional data saving
np.save("data1.npy", getattr(sim, "data1", np.zeros(1)))
np.save("data2.npy", getattr(sim, "data2", np.zeros(1)))
